In [1]:
from dataclasses import dataclass, replace
from datetime import date, timedelta
from typing import Optional, Dict, List, Tuple, Callable, TypeVar

import numpy as np
import matplotlib.pyplot as plt

from finstack import Money
from finstack.core.currency import USD
from finstack.core.market_data import MarketContext
from finstack.core.market_data.term_structures import DiscountCurve, HazardCurve, ForwardCurve
from finstack.core.cashflow import xirr
from finstack.core.dates.daycount import DayCount
from finstack.valuations.instruments import RevolvingCredit
from finstack.valuations.pricer import create_standard_registry

In [2]:

# Market data IDs
USD_DISC_ID = "USD.OIS"
SOFR_1M_ID = "USD.SOFR.1M"
BORROWER_HZD_ID = "BORROWER-HZD"

# Instrument parameters
COMMITMENT_AMOUNT = 5_000_000
DRAWN_AMOUNT = 1_500_000
COMMITMENT_DATE = date(2025, 1, 1)
MATURITY_DATE = date(2035, 1, 1)


@dataclass(frozen=True)
class FeeConfig:
    """Fee structure for revolving credit facility."""
    commitment_fee_bp: float = 25.0
    usage_fee_bp: float = 150.0  # Used in build_revolver_from
    facility_fee_bp: float = 0.0
    upfront_fee: float = 0.0


@dataclass(frozen=True)
class RateConfig:
    """Base rate and margin configuration."""
    margin_bp: float = 0.0
    # For IRR calculations when using fixed rate
    base_rate_annual: float = 0.055


@dataclass(frozen=True)
class UtilProcess:
    """Utilization process parameters (mean-reverting)."""
    target_rate: float = 0.25
    speed: float = 0.50
    volatility: float = 0.20


@dataclass(frozen=True)
class CreditAnchored:
    """Market-anchored credit spread process parameters."""
    hazard_curve_id: str = BORROWER_HZD_ID
    kappa: float = 0.50
    implied_vol: float = 0.25
    tenor_years: Optional[float] = None


@dataclass(frozen=True)
class McKnobs:
    """Monte Carlo simulation configuration."""
    recovery_rate: float = 0.40
    util_credit_corr: float = 0.80
    num_paths: int = 25000
    seed: int = 42
    util: UtilProcess = UtilProcess()
    credit: CreditAnchored = CreditAnchored()


# Default configurations
FEE_DEFAULT = FeeConfig()
RATE_DEFAULT = RateConfig()
MC_DEFAULT = McKnobs()

In [4]:

def build_market(as_of: date) -> MarketContext:
    """Create minimal market inputs: discount curve + borrower hazard curve + forward curve."""
    disc = DiscountCurve(
        USD_DISC_ID,
        as_of,
        [
            (0.0, 1.0),
            (1.0, 0.97),
            (3.0, 0.91),
        ],
    )

    # Constant hazard ~5% with 40% recovery
    hazard = HazardCurve(
        BORROWER_HZD_ID,
        as_of,
        [
            (1.0, 0.05),
            (5.0, 0.05),
        ],
        recovery_rate=0.40,
    )

    # SOFR 1M forward curve (tenor = 1/12 years)
    sofr_1m = ForwardCurve(
        SOFR_1M_ID,
        1.0 / 12.0,
        [
            (0.0, 0.0530),
            (1.0, 0.0550),
            (3.0, 0.0570),
        ],
        base_date=as_of,
    )

    market = MarketContext()
    market.insert_discount(disc)
    market.insert_forward(sofr_1m)
    market.insert_hazard(hazard)
    return market



In [3]:
def build_revolver_from(
    mc: McKnobs,
    fees: FeeConfig = FEE_DEFAULT,
    rate: RateConfig = RATE_DEFAULT,
) -> RevolvingCredit:
    """Create a credit-risky revolver from Monte Carlo configuration.
    
    Only the MC parameters vary across scenarios; all other instrument
    parameters are centralized above.
    """
    return RevolvingCredit.builder(
        instrument_id=f"REVOLVER_CR_{mc.credit.implied_vol:.2f}",
        commitment_amount=Money(COMMITMENT_AMOUNT, USD),
        drawn_amount=Money(DRAWN_AMOUNT, USD),
        commitment_date=COMMITMENT_DATE,
        maturity_date=MATURITY_DATE,
        base_rate_spec={
            "type": "floating",
            "index_id": SOFR_1M_ID,
            "margin_bp": rate.margin_bp,
            "reset_freq": "monthly",
        },
        payment_frequency="quarterly",
        fees={
            "commitment_fee_bp": fees.commitment_fee_bp,
            "usage_fee_bp": fees.usage_fee_bp,
            "facility_fee_bp": fees.facility_fee_bp,
            "upfront_fee": fees.upfront_fee,
        },
        draw_repay_spec={
            "stochastic": {
                "utilization_process": {
                    "type": "mean_reverting",
                    "target_rate": mc.util.target_rate,
                    "speed": mc.util.speed,
                    "volatility": mc.util.volatility,
                },
                "num_paths": mc.num_paths,
                "seed": mc.seed,
                "mc_config": {
                    "recovery_rate": mc.recovery_rate,
                    "credit_spread_process": {
                        "market_anchored": {
                            "hazard_curve_id": mc.credit.hazard_curve_id,
                            "kappa": mc.credit.kappa,
                            "implied_vol": mc.credit.implied_vol,
                            "tenor_years": mc.credit.tenor_years,
                        }
                    },
                    "util_credit_corr": mc.util_credit_corr,
                },
            }
        },
        discount_curve=USD_DISC_ID,
    )



In [18]:
def extract_arrays_from_dataset(
    ds, recovery_rate: float
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Convert captured PathDataset into numpy arrays: (times, util_paths, hazard_paths, pvs)."""
    paths = ds.paths
    if len(paths) == 0:
        return np.array([]), np.zeros((0, 0)), np.zeros((0, 0)), np.array([])
    
    times = np.array([pt.time for pt in paths[0].points], dtype=float)
    num_paths = len(paths)
    num_steps = len(paths[0])
    util = np.zeros((num_paths, num_steps))
    hazard = np.zeros((num_paths, num_steps))
    pvs = np.zeros((num_paths,))
    
    for i, p in enumerate(paths):
        pvs[i] = p.final_value
        for j, pt in enumerate(p.points):
            u = pt.get_var("spot") or 0.0
            cs = pt.get_var("credit_spread") or 0.0
            util[i, j] = u
            hazard[i, j] = cs / max(1.0 - recovery_rate, 1e-6)
    
    return times, util, hazard, pvs

In [5]:
as_of = COMMITMENT_DATE
market = build_market(as_of)
registry = create_standard_registry()

In [21]:
base_mc = replace(MC_DEFAULT, num_paths=10_000, seed=42)
base = build_revolver_from(base_mc)
mc = base.mc_paths(market, as_of=as_of, capture_mode="sample", sample_count=200, seed=42)

ds = mc.paths

In [28]:
import pandas as pd

times, util_paths, hazard_paths, pvs = extract_arrays_from_dataset(ds, 0.4)
df = pd.DataFrame({
    "times": times,
    "util_mean": util_paths.mean(axis=0),
    "hazard_mean": hazard_paths.mean(axis=0),
    #"pvs": pvs
})
#df["PV"] = pd.Series([None]*len(df))  # PV is per-path, not per-timestep; can summarize separately
df

,times,util_mean,hazard_mean
0,0.000000,0.300000,0.050000
1,0.244036,0.298768,0.050403
2,0.488072,0.298544,0.050827
3,0.732108,0.298828,0.051065
4,0.976144,0.297954,0.050984
5,1.220180,0.285481,0.050674
6,1.464217,0.292480,0.050992
7,1.708253,0.291969,0.050950
8,1.952289,0.287149,0.050262
9,2.196325,0.283976,0.050092


In [29]:
pvs

array([1343183.45, 2640660.72,  963897.86, 1437367.14, 3049330.24,
       1819305.22, 1351066.38, 3318705.09, 1332176.09, 1872278.19,
       1003367.44, 2994686.07,  969279.68, 2783207.79, 1513599.76,
       2124472.46, 2217647.83, 1335793.15, 2061872.7 , 1017922.38,
       2689966.08, 1506405.42, 2435592.16, 2511525.67, 2314587.1 ,
       1364856.9 , 3467177.72,  729460.02, 1402008.72, 3112817.67,
       2594685.09, 3602259.65,  627269.14, 1902414.56, 1186042.44,
       1838905.57,  550598.99,  474581.93, 1877674.64, 1552729.15,
        559539.31,  813037.15,  756025.43,  691977.18, 2867807.17,
       2499977.09, 2720172.77, 2916367.27,  657594.45, 1564159.54,
        766931.76, 2121195.76, 3266601.67, 1927409.81,  533510.33,
       2016386.23, 2313215.7 , 2441663.58, 1678828.74, 1727506.9 ,
       1568364.83,  240845.33,  805265.44, 1445277.76, 3003868.11,
        965370.13, 1241132.24, 1497217.82,  775015.74, 1456059.95,
       2619940.32, 1670684.48, 2303682.94, 1243693.6 ,  669867